<a href="https://colab.research.google.com/github/sugarfreecat/festival-musica/blob/main/ingestion/data_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instala as bibliotecas usadas no guia.
!pip install -q gspread faker

# Importações e Autenticação (Service Account)

In [3]:
# Importa as ferramentas para autenticação, geração de dados e sorteios.
import json, gspread
from google.colab import userdata
from faker import Faker
import random

# Lê a credencial salva nos Secrets e conecta à planilha.
info = json.loads(userdata.get("GCP_SA_JSON"))
gc = gspread.service_account_from_dict(info)
# Exibe o e-mail da conta autenticada.
print("Conectado como:", info["client_email"])

Conectado como: desafio1-python@festival-musica.iam.gserviceaccount.com


# Abrindo a Planilha

In [35]:
SHEET_ID = userdata.get("SHEET_ID")
NOME_TABELA = "atracoes"   # nome da tabela na ficha do tema, ex.: "vagas"

# Abre a planilha e seleciona sua primeira aba.
sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1
# Renomeia a aba e exibe os dados da planilha.
ws.update_title(NOME_TABELA)     # a aba passa a ter o nome da tabela
print(sh.title, "|", ws.title, "|", sh.url)

Desafio_01_Festival_Musica | atracoes | https://docs.google.com/spreadsheets/d/1Emgh_UWiwdayWKZUXvVGjANL7jMUnLDp0xAZ-jipxrw


# Cabeçalho - Tabela Dimensão

In [36]:
# Define as colunas que formarão o cabeçalho da tabela.
COLUNAS = ["atracao_id", "nome_artista_ficticio", "genero_musical", "edicao_festival", "dia_apresentacao", "palco", "horario_inicio"]

# Escreve o cabeçalho na primeira célula e confere o resultado.
ws.update(range_name="A1", values=[COLUNAS])
print("Cabeçalho confere:", ws.row_values(1) == COLUNAS)

Cabeçalho confere: True


# 10 Linhas de Teste - Tabela Dimensão

## Variáveis

In [37]:
NOMES_ARTISTAS_FICTICIOS = [
    # Indie / Alternative Rock
    "Velvet Television",
    "The Paper Moons",
    "Plastic Daydream",
    "Summer Funeral",
    "Cherry Static",
    "The Hollow Youth",
    "Velvet Cinema",
    "Sunday in Reverse",
    "Motel Flowers",
    "Pretty Little Wounds",
    "The Distant Rooms",
    "Satellite Hearts",
    "Bedroom Confessions",
    "Cigarette Summer",
    "The April Tapes",

    # Emo / Post-hardcore / Shoegaze
    "February Letters",
    "Glassjaw Garden",
    "Drowning Season",
    "Soft Collapse",
    "Porcelain Youth",
    "The Last Good Weekend",
    "Static Romance",
    "Blurry Memories",
    "Funeral for a Tuesday",
    "Basement Poets",
    "Daisy Chainsaw",
    "Faded Polaroids",
    "Everyone I Used to Know",
    "Bleeding Wallpaper",
    "The Unfinished Apologies",

    # Dream Pop / Ethereal / Gothic
    "Angel Static",
    "Ophelia's Ghost",
    "Velour",
    "Moonmilk",
    "The Sleeping Flowers",
    "Ghosts in Bloom",
    "Eternal September",
    "Heavenly Bodies",
    "Blue Funeral",
    "Soft Cathedral",
    "Lunar Veil",
    "The Violet Hour",
    "Dead Roses Club",
    "Dollhouse Reverie",
    "Saint Nobody",

    # Synthwave / Electronic / Darkwave
    "Neon Mourning",
    "Chrome Paradise",
    "Digital Ghost",
    "Midnight Protocol",
    "Cassette Dreams",
    "Afterimage",
    "Nightdrive.exe",
    "Future Funeral",
    "Synthetic Lovers",
    "Cyber Eden",
    "Velvet Circuit",
    "Nocturnal FM",
    "Static Angel",
    "Memory Leak",
    "Electric Nostalgia",

    # Vaporwave / Experimental / Glitch
    "Mallsoft Memories",
    "Pixel Funeral",
    "Internet Girlfriend",
    "404 Heaven",
    "Dream Sequence 98",
    "Virtual Girlfriend",
    "Plastic Heaven",
    "Windows to Nowhere",
    "Dial-Up Angel",
    "Aesthetic Corruption",
    "Empty Shopping Mall",
    "Soft Reset",
    "Dead Web",
    "Blue Screen Romance",
    "Paradise Offline",

    # Indie Pop / Bedroom Pop
    "Milk Teeth",
    "Peach Funeral",
    "Sugarcoma",
    "Honey Television",
    "Pillow Talk Club",
    "Pink Cigarettes",
    "The Sleepover",
    "Strawberry Static",
    "Little Deaths",
    "Sunday Crush",
    "Daisy Motel",
    "Cloudberry",
    "Lemon Hearts",
    "Plastic Kisses",
    "Almost Famous",

    # Punk / Garage / Grunge
    "Rotten Valentine",
    "The Cheap Thrills",
    "Dog Teeth",
    "Bad Reception",
    "Basement Fever",
    "Teenage Arson",
    "Dead Batteries",
    "The Wrong Crowd",
    "Trash Romance",
    "Acid Weekend",
    "Broken Television",
    "Filthy Little Angels",
    "Concrete Flowers",
    "Public Nuisance",
    "No Future Club",

    # Psychedelic / Progressive / Krautrock
    "Astral Motel",
    "The Cosmic Garden",
    "Velvet Comet",
    "Solar Children",
    "Electric Orchard",
    "Third Eye Cinema",
    "Planetarium",
    "Mushroom Television",
    "The Infinite Afternoon",
    "Cosmic Laundry",
    "Acid Cathedral",
    "Psychedelic Postcards",
    "Orbiting Strangers",
    "The Celestial Noise",
    "Mirage Machine",

    # Industrial / Noise / Avant-garde
    "Meat Computer",
    "Soft Violence",
    "Human Error",
    "Concrete Lullaby",
    "Machine Romance",
    "Pleasure Damage",
    "Body Language Error",
    "The Beautiful Ruins",
    "Industrial Princess",
    "Sickly Sweet",
    "Noisy Neighbors",
    "Synthetic Flesh",
    "Dead Format",
    "Violent Daydream",
    "The Uncanny Valley",

    # Folk / Slowcore / Singer-songwriter
    "Mourning Doves",
    "The Orchard Sleeps",
    "Wildflower Funeral",
    "November House",
    "Honey in the Wound",
    "The Quiet Disaster",
    "Pale Morning",
    "Bluebird Cemetery",
    "The Lonely Pines",
    "Ghost Town Hymns",
    "Wishing Well",
    "Lullaby for Strangers",
    "Cemetery Picnic",
    "Little Wooden Ghosts",
    "The Winter Diaries",

    # Hip-hop / Trip-hop / R&B
    "Velvet Syntax",
    "Lowkey Funeral",
    "Ghost Frequency",
    "Lilac Static",
    "Midnight Syntax",
    "Concrete Angels",
    "Digital Tears",
    "Velvet Underground.exe",
    "Blue Hour",
    "Phantom Traffic",
    "Sugar Damage",
    "Ghost Notes",
    "Cloud Palace",
    "Neon Bruises",
    "The Lonely Algorithm",

    # Dark Cabaret / Baroque Pop / Weird stuff
    "Madame Misfortune",
    "The Velvet Puppets",
    "Circus of Regrets",
    "Miss Dystopia",
    "The Unhappy Hour",
    "Dead Doll Orchestra",
    "Pretty Little Freakshow",
    "The Carnival Is Closed",
    "Cabaret Macabre",
    "The Mourning Sisters",
    "Dolly Funeral",
    "Saints of Nowhere",
    "The Peculiar People",
    "House of Oddities",
    "The Last Waltz Society",

    # Brasil alternativo
    "Jardim Elétrico",
    "Quarto Vazio",
    "Os Dias Estranhos",
    "Garotas de Plástico",
    "Clube da Meia-Noite",
    "Coração de Neon",
    "Festa no Fim do Mundo",
    "Pequenos Desastres",
    " Rádio Fantasma",
    "Veludo Azul",
    "As Flores Mortas",
    "Domingo Lunar",
    "Pânico no Paraíso",
    "Hotel Primavera",
    "Oceano de Vidro",
    "Cemitério de Borboletas",
    "Chuva Ácida",
    "Satélite Perdido",
    "Máquina de Sonhos",
    "Os Últimos Românticos",

    # Nomes de projetos solo / pseudônimos
    "Mimi Mourning",
    "Angel November",
    "Lola Static",
    "Dorian Bloom",
    "Violet.exe",
    "Sunday Saint",
    "Baby Funeral",
    "Mothgirl",
    "Cherry Venom",
    "Lunar Darling",
    "Dolly Disaster",
    "Ghostboy",
    "Cigarette Girl",
    "Velvet Valentine",
    "Opal Fever",
]

GENEROS_MUSICAIS = [
    # Rock & derivados
    "alternative rock",
    "indie rock",
    "post-rock",
    "psychedelic rock",
    "grunge",
    "hard rock",
    "punk rock",
    "post-punk",
    "new wave",
    "gothic rock",
    "pop rock",

    # Emo, punk & derivados
    "emo",
    "midwest emo",
    "emo pop",
    "screamo",
    "post-hardcore",
    "hardcore punk",
    "pop punk",
    "skate punk",
    "riot grrrl",
    "shoegaze",
    "dream pop",
    "slowcore",
    "sadcore",

    # Metal
    "heavy metal",
    "alternative metal",
    "nu metal",
    "gothic metal",

    # Indie & alternativo
    "indie",
    "indie pop",
    "bedroom pop",
    "lo-fi",
    "art pop",

    # Eletrônica
    "synthpop",
    "synthwave",
    "darkwave",
    "ambient",

    # Pop
    "art pop",
    "dream pop",
    "electropop",
    "dark pop",
    "indie pop",
    "synth-pop",
    "bubblegum pop",
    "power pop",
    "chamber pop",
    "psychedelic pop",
    "alternative pop",
    "experimental pop",
    "J-pop",
    "K-pop",
    "city pop",

    # Folk & acústico
    "folk",
    "indie folk",
    "dark folk",
    "freak folk",
    "psych folk",
    "neofolk",
    "folk rock",
    "dream folk",
    "singer-songwriter",
    "anti-folk",
    "americana",
    "country",
    "alt-country",
    "bluegrass",

    # Psicodélico & esotérico
    "psychedelia",
    "neo-psychedelia",
    "space ambient",
    "ethereal wave",
    "darkwave",
    "ritual ambient",
    "occult rock",
    "witch house",
    "ethereal",
    "cosmic disco",

    # Música brasileira
    "bossa nova",
    "rock brasileiro",
    "indie brasileiro",
    "pós-punk brasileiro",
    "psicodelia brasileira",

    # Outros cantos do mundo
    "shoegaze japonês",
    "city pop",
    "visual kei",
]

PALCOS = [
    "subsolo",
    "pista fantasma",
    "doma de vidro",
    "acustico",
]

EDICOES = [
    {"edicao": "Vol. I (2019)", "dias": ["12/04", "13/04", "14/04"]},
    {"edicao": "Vol. II (2020)", "dias": ["09/10", "10/10", "11/10"]},
    {"edicao": "Vol. III (2021)", "dias": ["15/10", "16/10", "17/10"]},
    {"edicao": "Vol. IV (2022)", "dias": ["21/04", "22/04", "23/04"]},
    {"edicao": "Vol. V (2023)", "dias": ["06/10", "07/10", "08/10"]},
    {"edicao": "Vol. VI (2024)", "dias": ["18/10", "19/10", "20/10"]},
    {"edicao": "Vol. VII (2025)", "dias": ["10/10", "11/10", "12/10"]},
    {"edicao": "Vol. VIII (2026)", "dias": ["16/10", "17/10", "18/10"]},
]

HORARIOS_INICIO = [
    "14:00",
    "19:00",
    "23:00",
    "03:00"
]

NOMES_ARTISTAS_FICTICIOS2 = [
    "Meu Romance Químico",
    "Cabeça de Rádio",
    "Poder Marítimo",
    "Namorada Venenosa",
    "Musa",
    "Perfurar o Véu",
    "Sukkubus",
    "All About Lily Chou-Chou",
    "Paranmaun",
    "SMI - Sua Majestade Infernal"
]

## Script

In [39]:
# Configura os geradores aleatórios para produzir resultados reproduzíveis.
fake = Faker("pt_BR")
Faker.seed(13)
random.seed(13)

def gerar_linha(i):
    edicao = random.choice(EDICOES) # escolhe uma edição aleatoriamente

    # Cria um registro fictício para as colunas definidas no esquema.
    return {
        "atracao_id": i,
        "nome_artista_ficticio": random.choice(NOMES_ARTISTAS_FICTICIOS2),
        "genero_musical": random.choice(GENEROS_MUSICAIS),
        "edicao_festival": "Festival Gatitos" + " " + edicao["edicao"],   # datas como texto
        "dia_apresentacao": random.choice(edicao["dias"]),   # datas como texto
        "palco": random.choice(PALCOS),
        "horario_inicio": random.choice(HORARIOS_INICIO),   # horas como texto
    }

# Gera 10 registros numerados de 1 a 10.
linhas = [gerar_linha(i) for i in range(1, 11)]

# Organiza os valores na ordem do cabeçalho; nome de coluna incorreto gera KeyError.
valores = [[linha[c] for c in COLUNAS] for linha in linhas]
# Acrescenta os registros à planilha e informa quantos foram enviados.
ws.append_rows(valores, value_input_option="RAW")
print(len(valores), "linhas enviadas")

10 linhas enviadas


# Conferir o Resultado na Planilha

In [40]:
# Lê todos os valores da aba para conferir o conteúdo enviado.
valores_lidos = ws.get_all_values()
# Exibe a quantidade de linhas, confere o cabeçalho e mostra exemplos.
print("Linhas (cabeçalho + dados):", len(valores_lidos))    # esperado: 11
print("Cabeçalho confere:", valores_lidos[0] == COLUNAS)
print(valores_lidos[:3])

Linhas (cabeçalho + dados): 11
Cabeçalho confere: True
[['atracao_id', 'nome_artista_ficticio', 'genero_musical', 'edicao_festival', 'dia_apresentacao', 'palco', 'horario_inicio'], ['1', 'Musa', 'sadcore', 'Gatitos Festival Vol. V (2023)', '08/10', 'pista fantasma', '19:00'], ['2', 'Poder Marítimo', 'hardcore punk', 'Gatitos Festival Vol. IV (2022)', '21/04', 'pista fantasma', '23:00']]


# Funções de Envio de Dados - Planilha

In [41]:
# Importa recursos para controlar pausas, acessar a planilha e tratar tabelas.
import time
import gspread
import pandas as pd

def df_para_valores(df):
    """DataFrame -> lista de listas com tipos nativos do Python."""
    # Trabalha em uma cópia para não alterar o DataFrame original.
    df = df.copy()
    for c in df.columns:
        # Converte datas para texto em um formato padronizado.
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.strftime("%Y-%m-%d %H:%M:%S")
        # Converte outros valores de data e hora para texto.
        elif df[c].dtype == object:
            df[c] = df[c].map(
                lambda v: v.isoformat() if hasattr(v, "isoformat") else v)
    # Substitui valores ausentes por texto vazio e retorna linhas de valores.
    return df.astype(object).where(df.notna(), "").values.tolist()

def enviar(sh, df, nome_aba, lote=5000, pausa=2):
    """Escreve cabeçalho e linhas numa aba, em lotes, e confere a contagem."""
    # Formata números com ponto como separador de milhar.
    n = lambda x: f"{x:,}".replace(",", ".")
    try:
        # Reutiliza a aba existente e apaga seu conteúdo anterior.
        ws = sh.worksheet(nome_aba)
        ws.clear()
    except gspread.WorksheetNotFound:
        # Cria uma aba caso ela ainda não exista.
        ws = sh.add_worksheet(title=nome_aba, rows=len(df) + 1,
                              cols=len(df.columns))
    # Ajusta o tamanho da aba e grava os nomes das colunas.
    ws.resize(rows=len(df) + 1, cols=len(df.columns))
    ws.update(range_name="A1", values=[list(df.columns)])
    # Converte o DataFrame e envia as linhas em lotes.
    valores = df_para_valores(df)
    for i in range(0, len(valores), lote):
        ws.append_rows(valores[i:i + lote], value_input_option="RAW")
        print(f"  {nome_aba}: {n(min(i + lote, len(valores)))} / {n(len(valores))}")
        time.sleep(pausa)               # respeita o limite de escritas/minuto
    # Confere se a quantidade de linhas gravadas corresponde à esperada.
    lidas = len(ws.col_values(1)) - 1   # lê de volta e confere
    if lidas != len(df):
        raise RuntimeError(f"Aba {nome_aba}: enviei {len(df)}, li {lidas}.")
    print(f"OK {nome_aba}: {n(len(df))} linhas na planilha")

def remover_aba_padrao(sh):
    """Apaga a aba vazia que o Google cria junto com a planilha nova."""
    # Localiza e remove a aba padrão somente se houver outra aba disponível.
    abas = sh.worksheets()
    padrao = ("Página1", "Sheet1", "Planilha1")
    for w in abas:
        if w.title in padrao and len(abas) > 1 and not w.get_all_values():
            sh.del_worksheet(w)

# Carregar Bases de Dados Prontos

In [42]:
# Abre a opção de selecionar arquivos CSV no Colab e envia os arquivos.
from google.colab import files
files.upload()   # selecione os arquivos CSV do seu tema

# Carrega cada CSV em um DataFrame e associa-o ao nome da tabela.
BASES = {
    "atracoes": pd.read_csv("atracoes.csv"),
    "ingressos_vendidos": pd.read_csv("ingressos_vendidos.csv"),
}
# Calcula e exibe a quantidade de linhas carregadas.
total = sum(len(d) for d in BASES.values())
print({n: len(d) for n, d in BASES.items()}, "| total:", total)

Saving ingressos_vendidos.csv to ingressos_vendidos.csv
Saving atracoes.csv to atracoes.csv
{'atracoes': 10100, 'ingressos_vendidos': 55000} | total: 65100


# Enviar Para a Planilha em Lotes

In [43]:
# Envia cada tabela para uma aba e remove a aba padrão, se estiver vazia.
for nome, df in BASES.items():
    enviar(sh, df, nome)        # uma aba por tabela
remover_aba_padrao(sh)

  atracoes: 5.000 / 10.100
  atracoes: 10.000 / 10.100
  atracoes: 10.100 / 10.100
OK atracoes: 10.100 linhas na planilha
  ingressos_vendidos: 5.000 / 55.000
  ingressos_vendidos: 10.000 / 55.000
  ingressos_vendidos: 15.000 / 55.000
  ingressos_vendidos: 20.000 / 55.000
  ingressos_vendidos: 25.000 / 55.000
  ingressos_vendidos: 30.000 / 55.000
  ingressos_vendidos: 35.000 / 55.000
  ingressos_vendidos: 40.000 / 55.000
  ingressos_vendidos: 45.000 / 55.000
  ingressos_vendidos: 50.000 / 55.000
  ingressos_vendidos: 55.000 / 55.000
OK ingressos_vendidos: 55.000 linhas na planilha


# Validar a Base na Planilha

In [44]:
# Soma e exibe a quantidade total de linhas das tabelas.
total = sum(len(d) for d in BASES.values())
print("Total de linhas:", f"{total:,}".replace(",", "."))
# Interrompe a execução se o volume mínimo não for atingido.
assert total >= 20_000, "Abaixo do mínimo (20.000 linhas somadas)."

# Confere os cabeçalhos e a quantidade de registros em cada aba.
for nome, df in BASES.items():
    w = sh.worksheet(nome)
    assert w.row_values(1) == list(df.columns), f"Cabeçalho diferente em {nome}"
    assert len(w.col_values(1)) - 1 == len(df), f"Contagem diferente em {nome}"
# Confirma que todas as verificações passaram.
print("A planilha confere com a base gerada.")

Total de linhas: 65.100
A planilha confere com a base gerada.


```markdown
# README.md

## 🇧🇷 Português

### Sobre o Projeto
Este projeto implementa e automatiza um pipeline profissional de ingestão, transformação e validação de dados para o banco de dados de eventos do **Festival Gatitos** (contendo as tabelas de `atrações` e `ingressos vendidos`). Utilizando a linguagem **Python**, a biblioteca **Pandas** para manipulação de dados e a biblioteca **gspread** integrada à API do Google Sheets/Drive, o sistema ingere com segurança mais de 65.000 registros na nuvem.

### 🚀 Destaques para Portfólio de Engenharia de Dados

* **Ingestão Baseada em Lotes (Batch Uploading & API Resilience):** Como o volume total supera 65.000 registros, um upload direto causaria interrupções devido aos limites de taxa da API da Google (*rate limiting*). Desenvolvemos um algoritmo de fracionamento em lotes (*chunking*) estruturado que gerencia dinamicamente o tráfego de rede e insere pausas inteligentes (`time.sleep`) para garantir a estabilidade do processo de ingestão.
* **Autenticação Segura e SecOps (IAM):** A integração com serviços do Google Cloud Platform (GCP) é estabelecida usando Contas de Serviço (*Service Accounts*). As chaves de acesso são manipuladas de forma restrita através do gerenciador de segredos nativo do ambiente (`google.colab.userdata`), evitando a prática insegura de expor credenciais diretamente no código-fonte.
* **Processamento e Tratamento de Dados (Pandas ETL):** Processamento eficiente de tipos de dados heterogêneos. Datas e horas nativas são padronizadas para strings compatíveis com as especificações da API Sheets, e valores ausentes (`NaN`) são devidamente preenchidos para manter a fidelidade e consistência analítica.
* **Garantia de Qualidade de Dados (Data Quality & Automation):** O fluxo é finalizado com rotinas de testes automatizados (`assert`) que comparam de forma rigorosa as assinaturas dos cabeçalhos das tabelas enviadas com o que foi fisicamente gravado no Google Sheets, assegurando que o volume de registros esteja 100% íntegro.

---

## 🇺🇸 English

### About the Project
This project implements and automates a professional data ingestion, transformation, and validation pipeline for the **Festival Gatitos** database (housing the `attractions` and `ticket sales` tables). Powered by **Python**, **Pandas** for data wrangling, and the **gspread** library connected to the Google Sheets/Drive APIs, this system safely synchronizes and validates over 65,000 live cloud records.

### 🚀 Key Portfolio Highlights

* **Batch Data Ingestion & API Resilience:** Ingesting over 65k rows directly risks throttling by Google's strict API rate limits. To overcome this infrastructure barrier, the pipeline features a custom batching algorithm that partitions data into manageable chunks, applying regulated pauses (`time.sleep`) to keep the ingestion flow reliable and error-free.
* **Secure Authentication & Cloud SecOps (IAM):** Connection to Google Cloud services is established using a private Service Account. Access keys are safely fetched through the environment's native secrets manager (`google.colab.userdata`), eliminating hardcoded credentials risk and showing robust cloud security alignment.
* **Data Wrangling & Transformation (Pandas ETL):** Advanced transformation layer built to parse heterogeneous data formats. Native date and time columns are cast to unified ISO strings, and missing data points (`NaN`) are resolved before synchronization to ensure downstream compatibility.
* **Automated Data Quality & Schema Validation:** Programmatic schema checks (`assert`) run automatically post-upload. The pipeline compares local schema signatures and row counts directly against cloud metadata, guaranteeing 100% end-to-end data integrity.
```